In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込み テキスト: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 25. テキスト — モデルテキスト テキスト テキスト [CPU/GPU ベンチマーク ⑪]

> **学習目標**
> - FP32, FP16, BF16, INT8, INT4テキスト 精度·テキスト·速度 テキスト テキスト
> - テキスト テキスト $q = \mathrm{round}(x/s) + z$テキスト テキスト度テキスト
> - GPTQ, AWQ, GGUF テキスト テキスト テキスト テキスト

## 25.1 テキスト テキスト

LLMテキスト テキスト·テキスト テキスト:
- LLaMA-7B FP16: 14GB
- 推論 テキスト テキスト KV Cache
- テキスト/テキスト テキスト テキスト

テキスト テキスト:
- FP16 → INT8: テキスト テキスト, 速度 2テキスト
- FP16 → INT4: テキスト 1/4, 速度 3-4テキスト

## 25.2 テキスト テキスト

| テキスト | テキスト | テキスト | テキスト | テキスト |
|---|---|---|---|---|
| FP32 | 1 | 8 | 23 | 32 bit |
| FP16 | 1 | 5 | 10 | 16 bit |
| BF16 | 1 | 8 | 7 | 16 bit |
| FP8 (E4M3) | 1 | 4 | 3 | 8 bit |

- FP16: 精度 テキスト テキスト テキスト (テキスト テキスト)
- BF16: FP32テキスト テキスト テキスト, 精度 テキスト → LLM 学習テキスト テキスト


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# テキスト テキスト 比較
print("テキストPoint テキスト テキスト:")
for name, dtype in [('FP32', torch.float32), ('FP16', torch.float16), ('BF16', torch.bfloat16)]:
    info = torch.finfo(dtype)
    print(f"  {name}: min={info.min:.2e}, max={info.max:.2e}, eps={info.eps:.2e}")

# FP16 テキスト テキスト
print("\nFP16 テキスト:")
print(f"  2^15 = {float(torch.tensor(2**15, dtype=torch.float16))}")
print(f"  2^16 = {float(torch.tensor(2**16, dtype=torch.float16))}  # inf!")
print(f"  BF16 2^16 = {float(torch.tensor(2**16, dtype=torch.bfloat16))}  # テキスト")


## 25.3 テキスト テキスト テキスト

FP32 値テキスト INT8 (0~255 テキスト -128~127)テキスト テキスト:

**テキスト テキスト** (zero-point = 0):
$$q = \mathrm{round}(x / s), \quad s = \frac{\max|x|}{127}$$
$$\hat{x} = s \cdot q$$

**テキスト テキスト** (zero-point テキスト):
$$q = \mathrm{round}(x / s) + z$$
$$s = \frac{\max(x) - \min(x)}{255}, \quad z = -\mathrm{round}(\min(x) / s)$$

テキスト 誤差: $\epsilon = x - \hat{x}$.


In [ ]:
# テキスト テキスト
def quantize_symmetric(x, n_bits=8):
    """Symmetric Quantization."""
    qmax = 2 ** (n_bits - 1) - 1  # 127 for INT8
    scale = x.abs().max() / qmax
    q = torch.round(x / scale).clamp(-qmax - 1, qmax).to(torch.int8)
    return q, scale

def dequantize_symmetric(q, scale):
    return q.to(torch.float32) * scale

# Test
torch.manual_seed(0)
x = torch.randn(100) * 5  # Mean 0, Standard Deviation 5
q, scale = quantize_symmetric(x, n_bits=8)
x_hat = dequantize_symmetric(q, scale)
error = (x - x_hat).abs()
print(f"テキスト テキスト: [{x.min():.3f}, {x.max():.3f}]")
print(f"Quantization テキスト: [{q.min()}, {q.max()}]")
print(f"テキスト: {scale:.6f}")
print(f"Mean Absolute error: {error.mean():.6f}")
print(f"Maximum Absolute error: {error.max():.6f}")
print(f"テキスト Error: {(error.mean() / x.abs().mean() * 100):.3f}%")

# テキスト テキスト テキスト 比較
print("\nテキスト テキスト Quantization Error:")
for n_bits in [8, 4, 2]:
    q, scale = quantize_symmetric(x, n_bits=n_bits)
    x_hat = dequantize_symmetric(q, scale)
    err = (x - x_hat).abs().mean()
    print(f"  INT{n_bits}: Mean Error = {err:.6f} ({err/x.abs().mean()*100:.2f}%)")


## 25.4 Per-tensor, Per-channel, Per-group

- **Per-tensor**: テキスト テキスト テキスト scale. テキスト テキスト度 テキスト.
- **Per-channel**: テキスト(テキスト/テキスト)テキスト scale. テキスト度 テキスト, テキスト テキスト テキスト.
- **Per-group**: $g$テキスト テキスト scale. テキスト. LLM テキスト テキスト.

テキスト: 4-bit テキスト group_size=128 → 128テキスト 値テキスト scale (FP16) テキスト.


In [ ]:
# Per-group テキスト
def quantize_per_group(x, n_bits=4, group_size=128):
    """テキスト Quantization."""
    qmax = 2 ** (n_bits - 1) - 1
    # テキスト テキスト
    orig_shape = x.shape
    x_flat = x.reshape(-1, group_size)
    scales = x_flat.abs().max(dim=1, keepdim=True).values / qmax
    q = torch.round(x_flat / scales).clamp(-qmax - 1, qmax)
    return q, scales, orig_shape

def dequantize_per_group(q, scales, orig_shape):
    return (q * scales).reshape(orig_shape)

# Comparison: per-tensor vs per-group
torch.manual_seed(0)
W = torch.randn(256, 256) * 0.1  # Weight Matrix

# per-tensor (4-bit)
qmax = 7  # 4-bit symmetric
scale_t = W.abs().max() / qmax
q_t = torch.round(W / scale_t).clamp(-8, 7)
W_hat_t = q_t * scale_t
err_t = (W - W_hat_t).abs().mean()

# per-group (4-bit, group=128)
q_g, scales_g, shape = quantize_per_group(W, n_bits=4, group_size=128)
W_hat_g = dequantize_per_group(q_g, scales_g, shape)
err_g = (W - W_hat_g).abs().mean()

print(f"4-bit テキスト 誤差:")
print(f"  Per-tensor: {err_t:.6f} ({err_t/W.abs().mean()*100:.2f}%)")
print(f"  Per-group (128): {err_g:.6f} ({err_g/W.abs().mean()*100:.2f}%)")
print("\n=> Per-groupテキスト テキスト テキスト. LLM Quantization Standard.")


## 25.5 PTQ (Post-Training Quantization) vs QAT

**PTQ**: 学習 テキスト テキスト. テキスト テキスト度 テキスト テキスト.

**QAT (Quantization-Aware Training)**: テキスト テキスト 学習. テキスト テキスト テキスト.

LLMテキスト テキスト PTQ テキスト (学習 テキスト テキスト テキスト).

## 25.6 GPTQ, AWQ

### GPTQ
- 2テキスト 誤差 テキスト テキスト
- テキスト テキスト (テキスト テキスト)
- Hessian 行列 テキスト 誤差 テキスト
- INT4テキスト度 PPL テキスト テキスト テキスト

### AWQ (Activation-aware Weight Quantization)
- テキスト テキスト(テキスト テキスト)テキスト FP16 テキスト
- テキスト値 テキスト テキスト度 テキスト
- GPTQテキスト テキスト テキスト


## 25.7 QLoRA — 4-bit テキスト + LoRA

QLoRA (Dettmers et al., 2023):
1. テキスト モデルテキスト 4-bit NF4テキスト テキスト (テキスト)
2. LoRA テキスト FP16/BF16テキスト 学習
3. テキスト: 7B モデルテキスト テキスト 24GB GPUテキスト テキスト テキスト

NF4 (NormalFloat 4-bit): テキスト テキスト分布テキスト テキスト テキスト テキスト 4-bit テキスト.


In [ ]:
# QLoRA テキスト テキスト
def qlora_memory(base_params_b, adapter_ratio=0.01):
    """QLoRA Memory Estimation (GB).
    base_params_b: テキスト Parameter Count (テキスト: 10テキスト)
    adapter_ratio: テキスト テキスト Ratio
    """
    # テキスト: 4-bit = 0.5 bytes
    base_mem = base_params_b * 0.5
    # テキスト: FP16 + AdamW (4テキスト) = 8 bytes
    adapter_mem = base_params_b * adapter_ratio * 8
    # テキストValue (テキスト)
    activation = 2  # GB
    return base_mem + adapter_mem + activation

# Comparison: テキスト テキスト vs QLoRA
print(f"{'Model':>10} | {'テキスト FT FP16 (GB)':>15} | {'QLoRA 4-bit (GB)':>17}")
print("-" * 50)
for name, n_b in [('7B', 7), ('13B', 13), ('30B', 30), ('70B', 70)]:
    full = n_b * 2 * 4 + 4  # FP16 + AdamW FP32 (4テキスト) + テキスト
    qlora = qlora_memory(n_b, 0.01)
    print(f"{name:>10} | {full:>15.1f} | {qlora:>17.1f}")
print("\n=> QLoRAテキスト 70B Model度 テキスト 48GB GPUテキスト テキスト テキスト.")


## 25.8 [CPU/GPU ベンチマーク ⑪] 精度テキスト 推論


In [ ]:
# 精度テキスト 推論 速度 比較
from llm_math.bench import time_fn

# テキスト テキスト 推論
def bench_linear(d_in, d_out, batch_size, dtype, device='cpu'):
    """LinearLayer Inference Time."""
    model = nn.Linear(d_in, d_out).to(dtype=dtype, device=device)
    x = torch.randn(batch_size, d_in, dtype=dtype, device=device)
    def run():
        with torch.no_grad():
            return model(x)
    return run

print(f"{'dtype':>10} | {'CPU (ms)':>10} | {'Memory (MB)':>12}")
print("-" * 40)
d_in, d_out, bs = 4096, 4096, 8
for name, dtype in [('FP32', torch.float32), ('FP16', torch.float16)]:
    if dtype == torch.float16 and not torch.cuda.is_available():
        # CPUテキスト FP16テキスト テキスト
        continue
    try:
        run = bench_linear(d_in, d_out, bs, dtype, 'cpu')
        # warmup
        run()
        res = time_fn(run, device='cpu', warmup=2, repeat=5)
        # Memory (テキスト)
        mem = d_in * d_out * (4 if dtype == torch.float32 else 2) / 1024**2
        print(f"{name:>10} | {res['mean_ms']:>10.3f} | {mem:>12.2f}")
    except Exception as e:
        print(f"{name:>10} | {'N/A':>10} | error: {e}")

print("\n=> FP16テキスト Memory テキスト, CPUテキスト Speed Improvement テキスト (GPUテキスト テキスト Improvement).")


## 25.9 要点

| テキスト | テキスト | テキスト テキスト | テキスト度 テキスト |
|---|---|---|---|
| FP16 | 16 | 2x | テキスト テキスト |
| BF16 | 16 | 2x | テキスト テキスト |
| INT8 | 8 | 4x | テキスト |
| INT4 (GPTQ/AWQ) | 4 | 8x | テキスト |
| QLoRA (NF4) | 4 | 8x | テキスト |

## 演習問題

1. INT8 テキスト テキスト テキスト 出力テキスト MSEテキスト テキスト.
2. Per-tensor vs Per-channel vs Per-group テキスト 誤差テキスト 比較テキスト.
3. テキスト テキスト 32, 64, 128, 256テキスト テキスト テキスト 誤差テキスト 比較テキスト.
4. FP32 vs FP16 推論 速度テキスト CPUテキスト GPUテキスト テキスト 比較テキスト.
5. QLoRAテキスト テキスト テキスト 比較テキスト テキスト テキスト テキスト 7B, 13B, 70B モデルテキスト テキスト 計算テキスト.

> 解答: `solutions/ch25_solutions.ipynb`
